In [5]:
import numpy as np
import pandas as pd

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
np.set_printoptions(suppress=True)

stats = pd.read_csv("data/advanced_stats_clean.csv")
salary = pd.read_csv("data/player_salary_clean.csv")

In [7]:
# join advanced stats and salary

def _norm(s: pd.Series) -> pd.Series:
    return s.astype(str).str.strip()

stats_j = stats.copy()
sal_j = salary.copy()
stats_j["_player"] = _norm(stats_j["Player"])
stats_j["_team"] = _norm(stats_j["Team"])
sal_j["_player"] = _norm(sal_j["Player"])
sal_j["_team"] = _norm(sal_j["Team"])

salary_extra_cols = [c for c in sal_j.columns if c not in ("Player", "Team", "_player", "_team")]
merged = stats_j.merge(
    sal_j[["_player", "_team"] + salary_extra_cols],
    on=["_player", "_team"],
    how="inner",
)
merged = merged.drop(columns=["_player", "_team"])

print("Stats rows:", len(stats))
print("Salary rows:", len(salary))
print("Merged rows (inner join on Player + Team):", len(merged))

Stats rows: 736
Salary rows: 530
Merged rows (inner join on Player + Team): 284


In [ ]:
# prediction target and table

# which season do we predict the salary for
TARGET_SALARY_COL = "2025-26"

# disregarding players who play less than 10 games or 0 minutes
MIN_GAMES = 10
MIN_MP = 0  

df = merged.copy()

if TARGET_SALARY_COL not in df.columns:
    sal_like = [c for c in df.columns if isinstance(c, str) and (c[:2] == "20" or c == "Guaranteed")]
    raise ValueError(f"Missing salary column {TARGET_SALARY_COL!r}. Candidates: {sal_like}")

df = df.rename(columns={TARGET_SALARY_COL: "salary"})

# rows with a salary for this season
before = len(df)
df = df[df["salary"].notna()].copy()
df = df[df["salary"] > 0]

if MIN_GAMES > 0 and "G" in df.columns:
    df = df[df["G"] >= MIN_GAMES]
if MIN_MP > 0 and "MP" in df.columns:
    df = df[df["MP"] >= MIN_MP]

# Salaries span huge highs vs lows; log scale evens that out for regression.
df["log_salary"] = np.log(df["salary"])

print("Target column:", TARGET_SALARY_COL, "-> salary")
print("Rows after dropping missing salary:", len(df), "(dropped", before - len(df), "with no salary)")
print("Salary describe:")
print(df["salary"].describe())
df.head()

Target column: 2025-26 -> salary
Rows after dropping missing salary: 273 (dropped 11 with no salary)
Salary describe:
count    2.730000e+02
mean     1.439503e+07
std      1.480150e+07
min      6.239670e+05
25%      2.966760e+06
50%      8.382150e+06
75%      1.950000e+07
max      5.960682e+07
Name: salary, dtype: float64


,Player,Age,Team,Pos,G,GS,MP,PER,TS%,3PAr,...,Rk,salary,2026-27,2027-28,2028-29,2029-30,2030-31,Guaranteed,player_id,log_salary
0,Mikal Bridges,28.0,NYK,SF,82.0,82.0,3036.0,14.0,0.585,0.391,...,77,24900000.0,33482145.0,36160714.0,38839285.0,41517856.0,NaN,133382144.0,bridgmi01,17.030378
1,Josh Hart,29.0,NYK,SG,77.0,77.0,2897.0,16.5,0.611,0.327,...,93,19472240.0,20923760.0,22375280.0,NaN,NaN,NaN,40396000.0,hartjo01,16.784500
2,Anthony Edwards,23.0,MIN,SG,79.0,79.0,2871.0,20.1,0.595,0.503,...,23,45550512.0,48924624.0,52298736.0,55672848.0,NaN,NaN,202446720.0,edwaran01,17.634332
3,Devin Booker,28.0,PHO,SG,75.0,75.0,2795.0,19.3,0.589,0.388,...,10,53142264.0,57078728.0,61015192.0,64065952.0,69191228.0,NaN,235302136.0,bookede01,17.788483
4,DeMar DeRozan,35.0,SAC,SF,77.0,77.0,2768.0,17.7,0.569,0.196,...,78,24570000.0,25740000.0,NaN,NaN,NaN,NaN,24570000.0,derozde01,17.017037


In [4]:
# Save merged table with target for modeling in a later notebook or script
out_path = "data/modeling_dataset.csv"
df.to_csv(out_path, index=False)
print("Wrote", out_path, "shape:", df.shape)

Wrote data/modeling_dataset.csv shape: (273, 38)
